### Parking payment transactions dataset (overview)

### 0. Summary of the dataset

#### What this dataset is
A time-series of paid parking activity aggregated into **10-minute bins**, grouped by **zone**, and split by **payment channel** (**meter** vs **mobile**) over **2014–2019**.

#### What information it contains
For each **zone** and **10-minute window** (start → end, with utc_start available), the data includes:
- **Meter channel:** meter_transactions, meter_payments
- **Mobile channel:** mobile_transactions, mobile_payments

Where:
- **transactions** = number of purchases in that time bin
- **payments** = total amount collected in that time bin

#### Adjustment made and why
I reshaped the dataset from **wide** (separate meter/mobile columns) to **long/tidy** format by creating:
- transaction_type ∈ {meter, mobile}
- amount (payments for that channel and time bin)
- n_transaction (transaction count for that channel and time bin)

**Why:** this makes the data easier to analyze and visualize because each column becomes a variable and each row becomes a single observation:
**(zone, time bin, channel)** — allowing simple grouping, comparisons of meter vs mobile, and consistent filtering.

### 1. Dataset Overview

Pivot: New columns -> transaction_type, transaction_amount

In [1]:
import polars as pl

payment_transaction = pl.read_csv("raw_data/Payment Transactions across meters.csv",schema_overrides={"meter_payments": pl.Float64})
payment_transaction.head(3)

_id,zone,start,end,utc_start,meter_transactions,meter_payments,mobile_transactions,mobile_payments
i64,str,str,str,str,i64,f64,i64,f64
1,"""421 - NorthSide""","""2018-01-01T00:20:00""","""2018-01-01T00:30:00""","""2018-01-01T05:20:00""",0,0.0,1,4.0
2,"""403 - Uptown""","""2018-01-01T01:10:00""","""2018-01-01T01:20:00""","""2018-01-01T06:10:00""",0,0.0,1,3.0
3,"""412 - East Liberty""","""2018-01-01T01:10:00""","""2018-01-01T01:20:00""","""2018-01-01T06:10:00""",0,0.0,1,3.0


Transform date to polar date format

In [2]:
import polars as pl

payment_transaction_transformed = payment_transaction.with_columns(
    [
        pl.col("start")
          .str.strptime(pl.Datetime, format="%Y-%m-%dT%H:%M:%S", strict=False)
          .alias("start"),

        pl.col("end")
          .str.strptime(pl.Datetime, format="%Y-%m-%dT%H:%M:%S", strict=False)
          .alias("end"),

        pl.col("utc_start")
          .str.strptime(pl.Datetime, format="%Y-%m-%dT%H:%M:%S", strict=False)
          .alias("utc_start"),
    ]
)

payment_transaction_transformed.sort("start").head(3)

_id,zone,start,end,utc_start,meter_transactions,meter_payments,mobile_transactions,mobile_payments
i64,str,datetime[μs],datetime[μs],datetime[μs],i64,f64,i64,f64
1399109,"""357 - Shiloh Street Lot""",2014-01-02 00:30:00,2014-01-02 00:40:00,2014-01-02 05:30:00,1,1.0,0,0.0
1399110,"""402 - Downtown 2""",2014-01-02 01:30:00,2014-01-02 01:40:00,2014-01-02 06:30:00,1,1.0,0,0.0
1399111,"""402 - Downtown 2""",2014-01-02 01:40:00,2014-01-02 01:50:00,2014-01-02 06:40:00,1,2.0,0,0.0


In [3]:
payment_transaction_transformed.describe()

statistic,_id,zone,start,end,utc_start,meter_transactions,meter_payments,mobile_transactions,mobile_payments
str,f64,str,str,str,str,f64,f64,f64,f64
"""count""",2.56e6,"""2560000""","""2560000""","""2560000""","""2560000""",2.56e6,2.56e6,2.56e6,2.56e6
"""null_count""",0.0,"""0""","""0""","""0""","""0""",0.0,0.0,0.0,0.0
"""mean""",1280000.5,null,"""2016-12-29 23:23:25.477265""","""2016-12-29 23:33:25.477265""","""2016-12-30 03:42:13.900078""",7.245232,15.672085,2.20258,5.980584
"""std""",739008.4889,null,null,null,null,9.486956,23.555337,4.47276,14.939453
"""min""",1.0,"""301 - Sheridan Harvard Lot""","""2014-01-02 00:30:00""","""2014-01-02 00:40:00""","""2014-01-02 05:30:00""",0.0,0.0,0.0,0.0
"""25%""",640001.0,null,"""2014-10-30 11:00:00""","""2014-10-30 11:10:00""","""2014-10-30 15:00:00""",1.0,1.75,0.0,0.0
"""50%""",1.280001e6,null,"""2018-02-23 18:20:00""","""2018-02-23 18:30:00""","""2018-02-23 23:20:00""",3.0,6.0,0.0,0.0
"""75%""",1.92e6,null,"""2018-12-01 14:50:00""","""2018-12-01 15:00:00""","""2018-12-01 19:50:00""",10.0,19.75,2.0,4.0
"""max""",2.56e6,"""427 - Knoxville""","""2019-09-04 11:50:00""","""2019-09-04 12:00:00""","""2019-09-04 15:50:00""",117.0,425.75,498.0,530.25


**Non fields level documentation found for this dataset**

Anyways, we can reasonably believe:   

Zone / grouping: zone (parking zone / lot, or “sampling zone” depending on the resource)   

Time window (aggregation bin):
* start (start timestamp of the bin, local time)
* end (end timestamp of the bin, local time)  

Time normalization:
* utc_start (same as start, expressed in UTC)

Meter activity (physical meters):
* meter_transactions (count of meter transactions in the bin)
* meter_payments (total $ paid at meters in the bin)   

Mobile activity (app payments):
* mobile_transactions (count of mobile transactions in the bin)
* mobile_payments (total $ paid via mobile in the bin)

**Check bin size**

In [4]:
tmp_data = payment_transaction_transformed.with_columns(
    ((pl.col("end") - pl.col("start")).dt.total_minutes()).alias("bin_minutes")
)
# see the most common bin sizes
print(tmp_data.group_by("bin_minutes").len())

# 10 minutes bins

shape: (1, 2)
┌─────────────┬─────────┐
│ bin_minutes ┆ len     │
│ ---         ┆ ---     │
│ i64         ┆ u32     │
╞═════════════╪═════════╡
│ 10          ┆ 2560000 │
└─────────────┴─────────┘


**Check the range of start and end time**

In [5]:
"""
Potential questions to explore:
1. How do parking meter payment patterns vary by location and time of day?
2. Which bins have highest average payment amounts?
3. Trends of payment through mobile app vs. card over time?
"""
# Check the range of start and end time
payment_transaction_transformed.select([
    pl.col("start").min().alias("start_min"),
    pl.col("start").max().alias("start_max"),
    pl.col("end").min().alias("end_min"),
    pl.col("end").max().alias("end_max"),
])

# The data covers from 2 Jan 2014 to 4 Sep 2019

start_min,start_max,end_min,end_max
datetime[μs],datetime[μs],datetime[μs],datetime[μs]
2014-01-02 00:30:00,2019-09-04 11:50:00,2014-01-02 00:40:00,2019-09-04 12:00:00


**Are there any rows that have both meter_payments and mobile payments = 0?**

In [6]:
# Are there any rows that have both meter_payments and mobile payments = 0?
payment_transaction.filter(
    (pl.col("meter_payments") == 0) &
    (pl.col("mobile_payments") == 0)
).height

# No, there aren't

0

In [7]:
# Are there any rows that have meter_transactions and mobile_transactions both > 0?
payment_transaction.filter(
    (pl.col("meter_transactions") > 0) &
    (pl.col("mobile_transactions") > 0)
).height

823545

In [8]:
# check whether start time values are unique
print("Number of unique start times:",payment_transaction.select("start").unique().height)
payment_transaction.select("start").unique().height == payment_transaction.height

Number of unique start times: 107114


False

### 2. Tidy Data

Pivot: New columns -> transaction_type, transaction_amount, n_transaction

In [9]:
pivoted = (
    payment_transaction
    # Create two STRUCT columns (one for meter, one for mobile).
    # Each struct packs the same logical fields:
    # - transaction_type (label)
    # - amount (payments)
    # - n_transaction (number of transactions)
    .with_columns([
        pl.struct([
            pl.lit("meter").alias("transaction_type"),          # label this record as "meter"
            pl.col("meter_payments").alias("amount"),           # payment amount for meter
            pl.col("meter_transactions").alias("n_transaction"),  # transaction count for meter
        ]).alias("meter_struct"),

        pl.struct([
            pl.lit("mobile").alias("transaction_type"),         # label this record as "mobile"
            pl.col("mobile_payments").alias("amount"),          # payment amount for mobile
            pl.col("mobile_transactions").alias("n_transaction"), # transaction count for mobile
        ]).alias("mobile_struct"),
    ])
    # Combine the two struct columns into a single LIST column.
    # Each row now contains a list of 2 structs: [meter_struct, mobile_struct].
    .with_columns(
        pl.concat_list(["meter_struct", "mobile_struct"]).alias("tx_list")
    )
    # Explode the list so each struct becomes its own row.
    # Result: each original row turns into 2 rows (meter + mobile).
    .explode("tx_list")
    # Expand (unnest) the struct fields into normal columns:
    # transaction_type, amount, n_transaction
    .unnest("tx_list")
    .drop(["meter_transactions", "meter_payments", "mobile_transactions",
           "mobile_payments", "meter_struct", "mobile_struct"])
    # Filter out rows that have 0 transactions
    .filter(
        pl.col("n_transaction") != 0
    )
)
pivoted.head()

_id,zone,start,end,utc_start,transaction_type,amount,n_transaction
i64,str,str,str,str,str,f64,i64
1,"""421 - NorthSide""","""2018-01-01T00:20:00""","""2018-01-01T00:30:00""","""2018-01-01T05:20:00""","""mobile""",4.0,1
2,"""403 - Uptown""","""2018-01-01T01:10:00""","""2018-01-01T01:20:00""","""2018-01-01T06:10:00""","""mobile""",3.0,1
3,"""412 - East Liberty""","""2018-01-01T01:10:00""","""2018-01-01T01:20:00""","""2018-01-01T06:10:00""","""mobile""",3.0,1
4,"""421 - NorthSide""","""2018-01-01T01:20:00""","""2018-01-01T01:30:00""","""2018-01-01T06:20:00""","""mobile""",4.0,1
5,"""402 - Downtown 2""","""2018-01-01T06:00:00""","""2018-01-01T06:10:00""","""2018-01-01T11:00:00""","""mobile""",16.25,1


### 3. Save filtered dataset

In [10]:
# save the cleaned data
pivoted.drop("_id").write_csv("../cleaned_data/payment_transaction_cleaned.csv")